# Part 1 — Environment setup, DYAMOND access, and IBTrACS ingestion

This notebook does three things:

1. Sets up the Python environment for the SciVis Task 2 workflow.
2. Verifies cloud access to the DYAMOND data through OpenVisus.
3. Downloads, cleans, and saves IBTrACS storm-track data for the DYAMOND time window.

Outputs created here:

- `data/raw/ibtracs/ibtracs.since1980.list.v04r01.csv.gz`
- `data/interim/event_catalog/ibtracs_points_20200120_20210320.parquet`
- `data/interim/event_catalog/ibtracs_storm_summary_20200120_20210320.parquet`

This notebook is intentionally conservative:
- small DYAMOND reads only
- no heavy extraction yet
- clean persistence for later notebooks

In [1]:
import sys
!{sys.executable} -m pip install --upgrade pip

!{sys.executable} -m pip install -r ../requirements.txt

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import pyspark

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("xarray:", xr.__version__)
print("PySpark:", pyspark.__version__)

NumPy: 1.26.4
Pandas: 2.2.2
xarray: 2024.3.0
PySpark: 3.5.1


In [3]:
from __future__ import annotations
from pathlib import Path
import matplotlib.pyplot as plt

In [4]:
# find proj root 
# should work whether the notebook is run from notebooks or from repo root

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    required = {"data", "src", "notebooks", "outputs", "README.md"}
    for candidate in candidates:
        names = {p.name for p in candidate.iterdir()} if candidate.exists() else set()
        if required.issubset(names):
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"

IBTRACS_RAW_DIR = RAW_DIR / "ibtracs"
EVENT_CATALOG_DIR = INTERIM_DIR / "event_catalog"

for path in [
    IBTRACS_RAW_DIR,
    EVENT_CATALOG_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/claudia/Downloads/UTK/COSC526/SciVis_2026')

In [6]:
from src.io.dyamond import (
    DYAMOND_BASE_URL,
    available_variables,
    dataset_summary,
    get_dataset_spec,
    open_dataset,
    read_logic_box,
)
from src.io.ibtracs import (
    download_ibtracs_csv,
    read_ibtracs_csv,
    clean_ibtracs,
    build_storm_summary,
    write_parquet,
)

## Configuration

DYAMOND info:
- DYAMOND is a coupled atmosphere–ocean simulation starting on 2020-01-20 and running for 14 months.
- Task 2 focuses on atmospheric `U, V, P, T` and ocean `Theta, u, v, salt`.

IBTrACS info:
- Use the `since1980` subset by default.
- Trim to the DYAMOND window to keep work focused and to avoid data size issues

In [7]:
CONFIG = {
    "dyamond_time_start": "2020-01-20",
    "dyamond_time_end": "2021-03-20",
    "ibtracs_subset": "since1980",
    "atmos_test_var": "U",
    "ocean_test_var": "Theta",
    "sample_quality": -6,   # coarse on purpose for a fast smoke test
    "sample_box_atmos": {
        "x": (0, 192),
        "y": (0, 192),
        "z": (0, 1),
        "time": None,
    },
    "sample_box_ocean": {
        "x": (0, 256),
        "y": (0, 256),
        "z": (0, 1),
        "time": None,
    },
}

CONFIG

{'dyamond_time_start': '2020-01-20',
 'dyamond_time_end': '2021-03-20',
 'ibtracs_subset': 'since1980',
 'atmos_test_var': 'U',
 'ocean_test_var': 'Theta',
 'sample_quality': -6,
 'sample_box_atmos': {'x': (0, 192), 'y': (0, 192), 'z': (0, 1), 'time': None},
 'sample_box_ocean': {'x': (0, 256), 'y': (0, 256), 'z': (0, 1), 'time': None}}

## Verify Environment

In [8]:
import OpenVisus as ov

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("OpenVisus imported successfully")
print("DYAMOND base URL:", DYAMOND_BASE_URL)
print("Atmos variables:", available_variables("atmos"))
print("Ocean variables:", available_variables("ocean"))

ModuleNotFoundError: No module named 'OpenVisus.VisusKernelPy'

## DYAMOND metadata access

This just opens the remote dataset metadata and prints the dataset shape and timestep count. There's no reason to load a massive amount of data for this.

In [ ]:
atmos_db = open_dataset("atmos", CONFIG["atmos_test_var"])
ocean_db = open_dataset("ocean", CONFIG["ocean_test_var"])

atmos_meta = dataset_summary(atmos_db)
ocean_meta = dataset_summary(ocean_db)

print("Atmospheric dataset summary")
print(atmos_meta)
print()
print("Ocean dataset summary")
print(ocean_meta)

## Small DYAMOND smoke-test reads

These are tiny on purpose. Just to verify remote access works before doing anything else.

In [ ]:
atmos_arr = read_logic_box(
    atmos_db,
    x=CONFIG["sample_box_atmos"]["x"],
    y=CONFIG["sample_box_atmos"]["y"],
    z=CONFIG["sample_box_atmos"]["z"],
    time=CONFIG["sample_box_atmos"]["time"],
    quality=CONFIG["sample_quality"],
)

ocean_arr = read_logic_box(
    ocean_db,
    x=CONFIG["sample_box_ocean"]["x"],
    y=CONFIG["sample_box_ocean"]["y"],
    z=CONFIG["sample_box_ocean"]["z"],
    time=CONFIG["sample_box_ocean"]["time"],
    quality=CONFIG["sample_quality"],
)

print("Atmos array shape:", getattr(atmos_arr, "shape", None), "dtype:", getattr(atmos_arr, "dtype", None))
print("Ocean array shape:", getattr(ocean_arr, "shape", None), "dtype:", getattr(ocean_arr, "dtype", None))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(np.squeeze(atmos_arr).T, origin="lower", aspect="auto")
axes[0].set_title(f'Atmos smoke test: {CONFIG["atmos_test_var"]}')

axes[1].imshow(np.squeeze(ocean_arr).T, origin="lower", aspect="auto")
axes[1].set_title(f'Ocean smoke test: {CONFIG["ocean_test_var"]}')

for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()

## IBTrACS download

Save the raw compressed CSV under `data/raw/ibtracs/` and keep the original file as is.

In [ ]:
ibtracs_raw_path = IBTRACS_RAW_DIR / "ibtracs.since1980.list.v04r01.csv.gz"

if not ibtracs_raw_path.exists():
    ibtracs_raw_path = download_ibtracs_csv(
        output_path=ibtracs_raw_path,
        subset=CONFIG["ibtracs_subset"],
    )

ibtracs_raw_path

## IBTrACS ingest and cleaning

Cleaning:
- parse `ISO_TIME` as UTC
- filter to the DYAMOND simulation window
- preserve both `lon_360` and `lon_180`
- turn wind and pressure from USA and WMO fields into `wind_kt` and `pres_mb`
- sort by storm ID and time

In [ ]:
ibtracs_raw = read_ibtracs_csv(ibtracs_raw_path)
ibtracs_points = clean_ibtracs(
    ibtracs_raw,
    start=CONFIG["dyamond_time_start"],
    end=CONFIG["dyamond_time_end"],
)

print("Raw rows:", len(ibtracs_raw))
print("Rows in DYAMOND window:", len(ibtracs_points))
print("Unique storms in DYAMOND window:", ibtracs_points["SID"].nunique())

ibtracs_points.head()

In [ ]:
display_cols = [
    "SID",
    "SEASON",
    "BASIN",
    "NAME",
    "ISO_TIME",
    "LAT",
    "LON",
    "lon_180",
    "lon_360",
    "wind_kt",
    "pres_mb",
    "NATURE",
    "TRACK_TYPE",
]

ibtracs_points[display_cols].sample(min(10, len(ibtracs_points)), random_state=42).sort_values(["SID", "ISO_TIME"])

## Storm summary table

Later notebooks can use this to pick storms and build an event catalog.

In [ ]:
storm_summary = build_storm_summary(ibtracs_points)

print("Storm summary rows:", len(storm_summary))
storm_summary.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

basin_counts = (
    storm_summary["basin"]
    .fillna("UNK")
    .value_counts()
    .sort_index()
)
axes[0].bar(basin_counts.index.astype(str), basin_counts.values)
axes[0].set_title("Storm counts by basin")
axes[0].set_xlabel("Basin")
axes[0].set_ylabel("Count")

max_wind = storm_summary["max_wind_kt"].dropna()
axes[1].hist(max_wind, bins=20)
axes[1].set_title("Distribution of storm max wind")
axes[1].set_xlabel("Wind (kt)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Store cleaned outputs

Save two parquet files:
- point-level track records for the DYAMOND period
- storm-level summary records for event selection

In [ ]:
points_out = EVENT_CATALOG_DIR / "ibtracs_points_20200120_20210320.parquet"
summary_out = EVENT_CATALOG_DIR / "ibtracs_storm_summary_20200120_20210320.parquet"

write_parquet(ibtracs_points, points_out)
write_parquet(storm_summary, summary_out)

print(points_out)
print(summary_out)